<a href="https://colab.research.google.com/github/yosie111/Shulchan_Aruch_RAG_1/blob/main/SA_RAG_Stage4_Generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Shulchan Aruch RAG — Stage 4: Generation + End-to-End Evaluation

## קלט:
- `shulchan_aruch_v2.jsonl` — chunks מ-Pipeline v2.1
- `על_שוע_חלק_א.xlsx` — 102 שאלות evaluation

## מה המחברת עושה:
1. טוענת chunks + בונה retriever (bge-m3 — המנצח מ-Stage 2)
2. מגדירה system prompt הלכתי עם citation enforcement
3. מחברת LLM (Claude API / OpenAI API / מקומי)
4. מריצה end-to-end: שאלה → retrieval → generation → תשובה מנומקת
5. מעריכה: citation accuracy, faithfulness, groundedness

## הוראות:
1. העלה JSONL + Excel
2. הכנס API key (תא 3)
3. הרץ תאים 1-6 ברצף


In [17]:
# ============================================================
# תא 1 — התקנה + ייבוא
# ============================================================
!pip install -q sentence-transformers chromadb openpyxl rank-bm25 anthropic openai

import json
import re
import os
import time
import numpy as np
from pathlib import Path
from collections import defaultdict

import openpyxl
import chromadb
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi

try:
    from google.colab import files as colab_files, userdata
    IN_COLAB = True
except Exception:
    IN_COLAB = False

import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
print('✅ תא 1')


Device: cuda
✅ תא 1


In [18]:
# ============================================================
# תא 2 — טעינת נתונים + בניית retriever (bge-m3)
# ============================================================

# --- A. טעינת chunks ---
JSONL_FILE = None
for p in sorted(Path('.').glob('*.jsonl')):
    JSONL_FILE = str(p); break
if JSONL_FILE is None and IN_COLAB:
    print('העלה JSONL:'); uploaded = colab_files.upload()
    for n in uploaded:
        if n.endswith('.jsonl'): JSONL_FILE = n; break
if not JSONL_FILE: raise FileNotFoundError('JSONL not found')

chunks = []
with open(JSONL_FILE, 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip(): chunks.append(json.loads(line))
content_chunks = [c for c in chunks if c.get('metadata',{}).get('doc_type') != 'hilchot_index']
print(f'📄 {len(content_chunks)} chunks')

# --- B. טעינת שאלות ---
EVAL_FILE = None
for p in sorted(Path('.').glob('*.xlsx')):
    EVAL_FILE = str(p); break
if EVAL_FILE is None and IN_COLAB:
    print('העלה Excel:'); uploaded = colab_files.upload()
    for n in uploaded:
        if n.endswith('.xlsx'): EVAL_FILE = n; break

GEMATRIA = {'א':1,'ב':2,'ג':3,'ד':4,'ה':5,'ו':6,'ז':7,'ח':8,'ט':9,
    'י':10,'כ':20,'ל':30,'מ':40,'נ':50,'ס':60,'ע':70,'פ':80,'צ':90,
    'ק':100,'ר':200,'ש':300,'ת':400,'ך':20,'ם':40,'ן':50,'ף':80,'ץ':90}
def heb2num(s):
    return sum(GEMATRIA.get(c,0) for c in re.sub(r'["\'\'  ׳״]','',s or ''))

eval_questions = []
if EVAL_FILE:
    wb = openpyxl.load_workbook(EVAL_FILE); ws = wb.active
    for row in range(2, ws.max_row + 1):
        q = ws.cell(row,1).value
        if not q: continue
        src = ws.cell(row,3).value or ''
        sm = re.search(r'סימן\s+([א-ת]+)', src)
        sem = re.search(r'סעיף\s+([א-ת]+)', src)
        eval_questions.append({
            'question': q,
            'answer': ws.cell(row,2).value or '',
            'source': src,
            'document': ws.cell(row,4).value or '',
            'gold_siman': heb2num(sm.group(1)) if sm else 0,
            'gold_seif': heb2num(sem.group(1)) if sem else 0,
        })
    print(f'📝 {len(eval_questions)} שאלות evaluation')

# --- C. Gold mapping ---
def find_gold_chunk(q, chs):
    gs, gse = q['gold_siman'], q['gold_seif']
    for c in chs:
        h = c.get('metadata',{}).get('hierarchy',{})
        if h.get('level_1_num') == gs:
            if gse in h.get('level_2_nums',[]) or gse == 0:
                return c['doc_id']
    for c in chs:
        h = c.get('metadata',{}).get('hierarchy',{})
        if h.get('level_1_num') == gs: return c['doc_id']
    return None

for q in eval_questions:
    q['gold_chunk_id'] = find_gold_chunk(q, content_chunks)

# --- D. Embedding model (bge-m3) ---
print('Loading bge-m3...')
embed_model = SentenceTransformer('BAAI/bge-m3', device=DEVICE)

chunk_ids = [c['doc_id'] for c in content_chunks]
chunk_texts_emb = [c['content']['text_for_embedding'] for c in content_chunks]
chunk_texts_llm = [c['content']['text_for_llm'] for c in content_chunks]
chunk_metadatas = [{'siman': c['metadata']['hierarchy']['level_1_num']} for c in content_chunks]

print('Encoding chunks...')
doc_vectors = embed_model.encode(chunk_texts_emb, show_progress_bar=True, normalize_embeddings=True)

# ChromaDB
chroma = chromadb.Client()
try: chroma.delete_collection('sa_rag')
except: pass
collection = chroma.create_collection('sa_rag', metadata={'hnsw:space': 'cosine'})
collection.add(
    ids=chunk_ids,
    embeddings=[v.tolist() for v in doc_vectors],
    documents=chunk_texts_llm,
    metadatas=chunk_metadatas,
)
print(f'✅ Vector DB: {collection.count()} chunks indexed')

# --- E. Retriever function ---
def retrieve(question, n_results=5):
    """שולף top-k chunks לשאלה נתונה"""
    qvec = embed_model.encode([question], normalize_embeddings=True)
    res = collection.query(
        query_embeddings=[qvec[0].tolist()],
        n_results=n_results,
    )
    results = []
    for i in range(len(res['ids'][0])):
        results.append({
            'chunk_id': res['ids'][0][i],
            'text': res['documents'][0][i],
            'distance': res['distances'][0][i] if res.get('distances') else None,
        })
    return results

# Test
test = retrieve('מתי מברכים על ציצית')
print(f'\nTest retrieval: {test[0]["chunk_id"]}')
print(f'✅ תא 2 — retriever מוכן')


📄 214 chunks
📝 102 שאלות evaluation
Loading bge-m3...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Encoding chunks...


Batches:   0%|          | 0/7 [00:00<?, ?it/s]

✅ Vector DB: 214 chunks indexed

Test retrieval: sa_oc_018_1-3
✅ תא 2 — retriever מוכן


In [19]:
# ============================================================
# תא 3 — הגדרת LLM (מודל מקומי)
# ============================================================

LLM_PROVIDER = 'local'

if LLM_PROVIDER == 'local':
    import torch
    from transformers import pipeline

    LLM_MODEL = 'dicta-il/dictalm2.0-instruct'
    print(f'Loading Local LLM: {LLM_MODEL}')

    # טעינת המודל - אם הוא כבר בזיכרון, זה יהיה מיידי
    llm_client = pipeline(
        'text-generation',
        model=LLM_MODEL,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        max_new_tokens=500,
        return_full_text=False
    )
    print(f'✅ LLM: Local ({LLM_MODEL}) loaded successfully!')

# ============================================================
# System Prompt — Aggressive Citation & Anti-Hallucination
# ============================================================

SYSTEM_PROMPT = """אתה מומחה הלכתי שעונה על שאלות אך ורק על סמך מקורות שסופקו לך.

כללים מחייבים:
1. ענה רק על סמך המקטעים שסופקו. אם התשובה לא נמצאת במקטעים — אמור "לא מצאתי מקור לכך במקטעים שסופקו".
2. בתחילת כל תשובה ציין את המקור המדויק: סימן + סעיף.
3. אם יש מחלוקת בין המחבר לרמ"א — ציין את שתי הדעות בנפרד.
4. השתמש בלשון הלכתית קצרה ומדויקת.
5. אל תוסיף מידע שאינו במקטעים, גם אם אתה "יודע" את התשובה.

פורמט תשובה:
מקור: [סימן X, סעיף Y]
תשובה: [תשובה קצרה]
פירוט: [הסבר קצר מבוסס על הטקסט]"""

def build_prompt(question, retrieved_chunks):
    """בניית prompt עם context מה-chunks"""
    context_parts = []
    for i, chunk in enumerate(retrieved_chunks, 1):
        context_parts.append(f'--- מקטע {i} ({chunk["chunk_id"]}) ---\n{chunk["text"]}')
    context = '\n\n'.join(context_parts)

    # מאחדים את הוראות המערכת ואת השאלה לטקסט אחד
    user_msg = f"""{SYSTEM_PROMPT}

להלן מקטעים מתוך שולחן ערוך:

{context}

---
שאלה: {question}"""
    return user_msg


def generate_answer(question, retrieved_chunks):
    """שולח שאלה + context ל-LLM ומקבל תשובה"""
    user_msg = build_prompt(question, retrieved_chunks)

    if LLM_PROVIDER == 'local':
        # שולחים רק הודעת user אחת כדי למנוע את השגיאה
        messages = [
            {"role": "user", "content": user_msg}
        ]

        result = llm_client(messages)
        return result[0]['generated_text'].strip()


# === Test ===
print('Testing RAG pipeline...')
test_q = 'מה צריך האדם לעשות בבוקר?'
test_chunks = retrieve(test_q, n_results=3)
test_answer = generate_answer(test_q, test_chunks)
print(f'\nQ: {test_q}')
print(f'Retrieved: {[c["chunk_id"] for c in test_chunks]}')
print(f'Answer:\n{test_answer}')
print(f'\n✅ תא 3 — RAG pipeline עובד')

Loading Local LLM: dicta-il/dictalm2.0-instruct


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


✅ LLM: Local (dicta-il/dictalm2.0-instruct) loaded successfully!
Testing RAG pipeline...

Q: מה צריך האדם לעשות בבוקר?
Retrieved: ['sa_oc_001_1-4__a', 'sa_oc_002_1-6', 'sa_oc_006_1__a']
Answer:
מקור: [סימן X, סעיף Y]
תשובה: לאחר ההשכמה בבוקר, האדם צריך להתפלל בהתלהבות, להתחנן לפני בוראו, ולשים לב לתפילות המיוחדות על חורבן בית המקדש והגלות (סימן א'). כמו גם, עליו ללבוש את בגדיו בזהירות, ללבוש חלוקו תוך שהוא שוכב, לנעול ולקשור את נעליו כראוי (סעיפים א'-ד'), להימנע מללכת בגילוי ראש ובלי כיסוי לכל גופו (סעיף ה'), ולבדוק את נקביו (סעיף ו'). הוא גם צריך להגיד את ברכת "אשר יצר" ואת "אלוהי נשמה" עם פירושים שונים, בהכרה בחכמה האלוהית בבריאת גוף האדם ותפקודו (סעיף א').

✅ תא 3 — RAG pipeline עובד


In [20]:
# ============================================================
# תא 4 — הרצת evaluation מלא על 102 שאלות
# ============================================================

N_RETRIEVE = 5  # כמה chunks לשלוף per שאלה

results = []
errors = []

print(f'Running {len(eval_questions)} questions (N={N_RETRIEVE})...\n')

for i, q in enumerate(eval_questions):
    try:
        # Retrieve
        retrieved = retrieve(q['question'], n_results=N_RETRIEVE)
        retrieved_ids = [r['chunk_id'] for r in retrieved]

        # Generate
        answer = generate_answer(q['question'], retrieved)

        # Store
        results.append({
            'idx': i,
            'question': q['question'],
            'gold_answer': q['answer'],
            'gold_source': q['source'],
            'gold_chunk_id': q.get('gold_chunk_id'),
            'retrieved_ids': retrieved_ids,
            'retrieval_hit': q.get('gold_chunk_id') in retrieved_ids,
            'generated_answer': answer,
        })

        # Progress
        hit = '✅' if q.get('gold_chunk_id') in retrieved_ids else '❌'
        if (i+1) % 10 == 0 or i < 3:
            print(f'  [{i+1:>3}/{len(eval_questions)}] {hit} {q["question"][:50]}')

    except Exception as e:
        errors.append((i, str(e)))
        print(f'  [{i+1:>3}] ❗ Error: {str(e)[:60]}')

print(f'\n✅ Done: {len(results)} answers, {len(errors)} errors')

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Running 102 questions (N=5)...



Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  [  1/102] ✅ מה צריך האדם לעשות בבוקר ?


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

  [ 90/102] ✅ כמה ברכות חייב אדם ביום ?


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:2 for op

  [100/102] ✅ האם מותר לשנות תפילין של ראש לשל יד?


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



✅ Done: 102 answers, 0 errors


In [23]:
# ============================================================
# תא 5 — מדדי הערכה
# ============================================================

def extract_cited_source(answer_text):
    """מחלץ סימן+סעיף מהתשובה"""
    m = re.search(r'סימן\s+([א-ת\'\"״׳]+)', answer_text)
    siman = m.group(1) if m else ''
    m2 = re.search(r'סעיף\s+([א-ת\'\"״׳]+)', answer_text)
    seif = m2.group(1) if m2 else ''
    return siman, seif


def check_citation_accuracy(result, q):
    """האם הציטוט שהLLM נתן תואם ל-gold?"""
    cited_siman, cited_seif = extract_cited_source(result['generated_answer'])
    gold_src = q.get('source', '')
    # בדוק אם הסימן שצוטט מופיע ב-gold source
    if cited_siman and cited_siman in gold_src:
        return 'correct'
    elif cited_siman:
        return 'wrong_citation'
    else:
        return 'no_citation'


def check_answer_match(result, q):
    """האם התשובה מכילה את תשובת ה-gold?"""
    gold = q.get('answer', '').strip()
    generated = result['generated_answer']
    if not gold: return 'no_gold'
    # Exact substring
    if gold in generated: return 'exact_match'
    # Word overlap
    gold_words = set(gold.split())
    gen_words = set(generated.split())
    overlap = len(gold_words & gen_words) / max(len(gold_words), 1)
    if overlap >= 0.5: return 'partial_match'
    return 'no_match'


def check_no_hallucination(result):
    """האם הLLM סירב כשלא מצא מקור?"""
    answer = result['generated_answer']
    refusal_phrases = ['לא מצאתי', 'לא נמצא', 'אין במקטעים', 'לא ניתן לענות']
    has_refusal = any(p in answer for p in refusal_phrases)
    has_retrieval = result['retrieval_hit']
    if has_retrieval and not has_refusal: return 'answered_correctly'
    if not has_retrieval and has_refusal: return 'refused_correctly'
    if not has_retrieval and not has_refusal: return 'hallucination_risk'
    if has_retrieval and has_refusal: return 'false_refusal'


# === חישוב מדדים ===
retrieval_hits = sum(1 for r in results if r['retrieval_hit'])

citation_stats = defaultdict(int)
answer_stats = defaultdict(int)
halluc_stats = defaultdict(int)

for r in results:
    q = eval_questions[r['idx']]
    citation_stats[check_citation_accuracy(r, q)] += 1
    answer_stats[check_answer_match(r, q)] += 1
    halluc_stats[check_no_hallucination(r)] += 1

n = len(results)
print(f'{"="*70}')
print(f'{"END-TO-END EVALUATION RESULTS":^70}')
print(f'{"="*70}')
print(f'  שאלות: {n}')
print(f'  LLM: {LLM_PROVIDER} ({LLM_MODEL})')
print()

print(f'--- Retrieval ---')
print(f'  Recall@{N_RETRIEVE}: {retrieval_hits/n:.3f} ({retrieval_hits}/{n})')
print()

print(f'--- Citation Accuracy ---')
for k in ['correct', 'wrong_citation', 'no_citation']:
    v = citation_stats[k]
    print(f'  {k:<20}: {v:>3} ({100*v//n}%)')
print()

print(f'--- Answer Match ---')
for k in ['exact_match', 'partial_match', 'no_match', 'no_gold']:
    v = answer_stats[k]
    print(f'  {k:<20}: {v:>3} ({100*v//n}%)')
print()

print(f'--- Hallucination Check ---')
for k in ['answered_correctly', 'refused_correctly', 'hallucination_risk', 'false_refusal']:
    v = halluc_stats[k]
    print(f'  {k:<20}: {v:>3} ({100*v//n}%)')

# Overall score
correct_citations = citation_stats['correct']
exact_answers = answer_stats['exact_match'] + answer_stats['partial_match']
safe = halluc_stats['answered_correctly'] + halluc_stats['refused_correctly']
print(f'\n{"="*70}')
print(f'  📊 Citation accuracy:  {100*correct_citations//n}%')
print(f'  📊 Answer match:       {100*exact_answers//n}%')
print(f'  📊 Safety (no halluc): {100*safe//n}%')
print(f'{"="*70}')

print(f'\n✅ תא 5')


                    END-TO-END EVALUATION RESULTS                     
  שאלות: 102
  LLM: local (dicta-il/dictalm2.0-instruct)

--- Retrieval ---
  Recall@5: 0.873 (89/102)

--- Citation Accuracy ---
  correct             :  40 (39%)
  wrong_citation      :  28 (27%)
  no_citation         :  34 (33%)

--- Answer Match ---
  exact_match         :  38 (37%)
  partial_match       :  17 (16%)
  no_match            :  47 (46%)
  no_gold             :   0 (0%)

--- Hallucination Check ---
  answered_correctly  :  81 (79%)
  refused_correctly   :   0 (0%)
  hallucination_risk  :  13 (12%)
  false_refusal       :   8 (7%)

  📊 Citation accuracy:  39%
  📊 Answer match:       53%
  📊 Safety (no halluc): 79%

✅ תא 5


In [22]:
# ============================================================
# תא 6 — דוגמאות + ניתוח שגיאות
# ============================================================

# --- A. דוגמאות מוצלחות ---
print(f'{"="*70}')
print(f'{"GOOD EXAMPLES":^70}')
print(f'{"="*70}')

good = [r for r in results if r['retrieval_hit'] and
        check_answer_match(r, eval_questions[r['idx']]) in ('exact_match', 'partial_match')]

for r in good[:5]:
    q = eval_questions[r['idx']]
    print(f'\nQ: {r["question"]}')
    print(f'Gold: {q["answer"]} ({q["source"]})')
    print(f'Generated:\n{r["generated_answer"][:200]}')
    print('-' * 50)

# --- B. כשלי retrieval ---
print(f'\n{"="*70}')
print(f'{"RETRIEVAL FAILURES":^70}')
print(f'{"="*70}')

ret_fail = [r for r in results if not r['retrieval_hit']]
for r in ret_fail[:5]:
    q = eval_questions[r['idx']]
    print(f'\nQ: {r["question"]}')
    print(f'Gold: {q["gold_chunk_id"]}')
    print(f'Got: {r["retrieved_ids"][:3]}')
    print(f'Answer: {r["generated_answer"][:150]}')
    print('-' * 50)

# --- C. Hallucination cases ---
print(f'\n{"="*70}')
print(f'{"HALLUCINATION RISK":^70}')
print(f'{"="*70}')

halluc_cases = [r for r in results if check_no_hallucination(r) == 'hallucination_risk']
if halluc_cases:
    for r in halluc_cases[:5]:
        q = eval_questions[r['idx']]
        print(f'\nQ: {r["question"]}')
        print(f'Gold chunk: {q["gold_chunk_id"]} (NOT retrieved)')
        print(f'Answer: {r["generated_answer"][:200]}')
        print('-' * 50)
else:
    print('\n  אין מקרי hallucination! 🎉')

# --- D. שמירת תוצאות ---
output = {
    'config': {
        'llm': f'{LLM_PROVIDER}/{LLM_MODEL}',
        'embedding': 'BAAI/bge-m3',
        'n_retrieve': N_RETRIEVE,
        'n_questions': len(results),
    },
    'metrics': {
        'retrieval_recall': retrieval_hits / n,
        'citation_accuracy': correct_citations / n,
        'answer_match': exact_answers / n,
        'safety': safe / n,
    },
    'results': results,
}

OUTPUT_DIR = '/content/pipeline_output' if IN_COLAB else 'pipeline_output'
os.makedirs(OUTPUT_DIR, exist_ok=True)
out_path = os.path.join(OUTPUT_DIR, 'rag_evaluation_results.json')
with open(out_path, 'w', encoding='utf-8') as f:
    json.dump(output, f, ensure_ascii=False, indent=2)
print(f'\n📄 Results saved: {out_path}')

if IN_COLAB:
    from google.colab import files as colab_files
    colab_files.download(out_path)

print(f'\n✅ תא 6 — סיום')


                            GOOD EXAMPLES                             

Q: מה צריך האדם לעשות בבוקר ?
Gold: יתגבר כארי (סימן א, סעיף א)
Generated:
לפי המידע שסופק, האדם צריך להתחיל את יומו בהשכמה, לומר את הפסוקים "שויתי ה' לנגדי תמיד" ו"יתגבר כארי לעמוד בבוקר לעבודת בוראו", להתפלל בזמן התפילה שהציבור מתפלל, ולברך ברכת השחר. הוא גם צריך להשתדל לע
--------------------------------------------------

Q: האם מותר לומר עשרת הדברות בציבור ?
Gold: רק ביחיד (סימן א, סעיף ה)
Generated:
מקור: [סימן X, סעיף Y]
תשובה: [לא מצאתי מקור לכך במקטעים שסופקו]

פירוט: שולחן ערוך אינו מתיר לומר עשרת הדברות בציבור (סימן 1, סעיף 5, וגם סימן 109, סעיף 6). הסיבה היא שעשרת הדברות נאמרים ביחיד בלבד, 
--------------------------------------------------

Q: איזו רגל חולץ תחילה ?
Gold: של שמאל תחילה (סימן ב, סעיף ה)
Generated:
סעיף ד משולחן ערוך (הלכות הנהגת אדם בבוקר, סימן ב): "ינעול מנעל ימין תחלה ולא יקשרנו, ואחר כך ינעול של שמאל ויקשרנו, ויחזור ויקשור של ימין."

תשובה: סעיף ד
-------------------------------------

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ תא 6 — סיום


בטח, הנה פירוט המלא של שלושת הפרומפטים (System Prompts) שהרצנו בניסוי, כפי שניסחנו אותם במהלך הבדיקות:

### 1. הפרומפט המקורי ("המומחה ההלכתי") - **הפרומפט המנצח 🏆**

זהו הפרומפט הראשון שהיה במחברת. הוא מאוזן, מגדיר את המודל כמומחה, ומציג 5 כללים פשוטים וברורים. התברר שהוא נתן את התוצאות הטובות ביותר (הכי הרבה ציטוטים מדויקים והכי פחות סירובי שווא).

```text
אתה מומחה הלכתי שעונה על שאלות אך ורק על סמך מקורות שסופקו לך.

כללים מחייבים:
1. ענה רק על סמך המקטעים שסופקו. אם התשובה לא נמצאת במקטעים — אמור "לא מצאתי מקור לכך במקטעים שסופקו".
2. בתחילת כל תשובה ציין את המקור המדויק: סימן + סעיף.
3. אם יש מחלוקת בין המחבר לרמ"א — ציין את שתי הדעות בנפרד.
4. השתמש בלשון הלכתית קצרה ומדויקת.
5. אל תוסיף מידע שאינו במקטעים, גם אם אתה "יודע" את התשובה.

פורמט תשובה:
מקור: [סימן X, סעיף Y]
תשובה: [תשובה קצרה]
פירוט: [הסבר קצר מבוסס על הטקסט]

```

---

### 2. הפרומפט האגרסיבי ("מערכת נוקשה")

זהו הפרומפט שנועד "להפחיד" את המודל כדי למנוע ממנו להזות מידע, באמצעות שפה מאיימת ("אזהרה חמורה", "שגיאה קריטית"). בפועל, הוא גרם למודל להיכנס למגננה, לזנק באחוזי סירוב השווא (מקרים בהם סירב לענות גם כשהתשובה הייתה מולו) ולאבד את הפורמט.

```text
אתה מערכת RAG הלכתית נוקשה ביותר. תפקידך היחיד הוא לחלץ מידע מדויק מהמקטעים המצורפים בלבד.

אזהרה חמורה:
אסור לך בשום אופן להשתמש בידע מוקדם, להסיק מסקנות עצמאיות או להשלים פערים מהזיכרון שלך!
אם השאלה אינה נענית במפורש בתוך המקטעים המצורפים שחור על גבי לבן, עליך לענות אך ורק את המילים: "לא מצאתי מקור לכך במקטעים שסופקו". המצאת תשובה או ציטוט מקור שאינו קיים במקטעים היא שגיאה קריטית שעוברת על חוקי המערכת.

כללים מחייבים:
1. קרא את המקטעים. האם התשובה לשאלה נמצאת שם במפורש? אם לא -> ענה "לא מצאתי מקור לכך במקטעים שסופקו" ועצור.
2. אם התשובה שם, העתק או סכם אותה במדויק מבלי להוסיף מילה משלך.
3. חובה להתחיל את התשובה בציון המקור המדויק (סימן + סעיף) כפי שהוא מופיע במקטע.
4. במקרה של מחלוקת בין המחבר לרמ"א (ורק אם היא מוזכרת במקטע) — ציין את שניהם.

פורמט תשובה קשיח (ואסור לחרוג ממנו):
מקור: [סימן X, סעיף Y]
תשובה: [תשובה קצרה מתוך הטקסט בלבד]
פירוט: [הסבר קצר שנשען אך ורק על המקטע]

```

---

### 3. הפרומפט החיובי והמובנה ("העוזר המדויק")

הפרומפט שנועד לתקן את נזקי הפרומפט האגרסיבי. הוא שמר על דרישות קשיחות (במיוחד כדי למנוע מהמודל להוסיף טקסט מקדים לפני הפורמט), אך השתמש בשפה חיובית ("התבסס אך ורק על...") במקום איומים. הוא שיפר מעט את המצב לעומת הפרומפט האגרסיבי, אך עדיין לא גבר על הפרומפט המקורי.

```text
אתה עוזר הלכתי מדויק ואמין. תפקידך לענות על שאלות אך ורק על בסיס מקטעי הטקסט מתוך השולחן ערוך שסופקו לך.

הנחיות פעולה:
1. התבסס אך ורק על המידע המופיע במקטעים. אל תשתמש בידע כללי או קודם.
2. אם התשובה לשאלה אינה מופיעה במפורש במקטעים, עליך לענות בדיוק כך: "לא מצאתי מקור לכך במקטעים שסופקו".
3. תמיד ציין את המקור המדויק (סימן וסעיף) כפי שהוא מופיע במקטע הנבחר.
4. נסח את התשובה בצורה הלכתית, קצרה ועניינית.

עליך להחזיר את התשובה אך ורק בפורמט הבא, ללא טקסט נוסף לפני או אחרי:
מקור: [סימן X, סעיף Y]
תשובה: [תשובה קצרה וברורה]
פירוט: [הסבר קצר מבוסס על הטקסט]

```

======================================================================
סיכום ניסוי Prompt Engineering - מערכת RAG הלכתית (שולחן ערוך)
======================================================================

1. רקע ומטרת הניסוי
--------------------------------------------------
* מודל שפה (LLM): dicta-il/dictalm2.0-instruct (הרצה מקומית)
* מודל שליפה (Retriever): BAAI/bge-m3 (עם Recall@5 של 87.3%)
* כמות שאלות במדגם: 102 שאלות
* מטרה: למצוא את ה-System Prompt שימקסם דיוק, ישמור על פורמט הלכתי קפדני (ציון סימן וסעיף), וימזער הזיות (Hallucinations) במקרים בהם המידע חסר.

2. הפרומפטים שנבדקו
--------------------------------------------------
א. הפרומפט המקורי ("המומחה ההלכתי"):
   גישה עניינית המגדירה את המודל כמומחה, מלווה ב-5 כללי "עשה ואל תעשה" פשוטים, ודוגמת פורמט קשיחה.

ב. הפרומפט האגרסיבי ("מערכת נוקשה"):
   גישה מאיימת הכוללת "אזהרה חמורה", הגדרת המצאת תשובות כ"שגיאה קריטית שעוברת על חוקי המערכת", והוראה נוקשה לענות "לא מצאתי" אם המידע לא קיים מילה במילה.

ג. הפרומפט החיובי/מובנה ("העוזר המדויק"):
   גישה סמכותית אך חיובית, נטולת איומים, שמתמקדת בהוראות "עשה" ("התבסס אך ורק על..."), תוך סגירה הרמטית של טקסט מיותר לפני ואחרי התשובה.

3. תוצאות ההערכה (End-to-End Evaluation)
--------------------------------------------------
| מדד (Metric)            | פרומפט 1 (מקורי) | פרומפט 2 (אגרסיבי) | פרומפט 3 (חיובי) |
|-------------------------|------------------|--------------------|------------------|
| דיוק ציטוט (Citation)   | 46%              | 37%                | 39%              |
| התאמת תשובה (Answer)    | 62%              | 51%                | 53%              |
| בטיחות כוללת (Safety)   | 85%              | 73%                | 79%              |
| סירובי שווא (False Ref) | 2%               | 14%                | 7%               |
| סכנת הזיה (Hallucination)| 11%             | 11%                | 12%              |

4. מסקנות ותובנות מרכזיות
--------------------------------------------------
* המנצח הברור: הפרומפט המקורי מנצח בכל המדדים בצורה מובהקת.
* עומס קוגניטיבי ואיומים: הפעלת לחץ (פרומפט אגרסיבי) שברה את המודל. היא גרמה לזינוק בסירובי שווא (מ-2% ל-14%) וגרמה לו לאבד את פורמט הציטוט שנדרש ממנו (צניחה ל-37% בציטוטים נכונים). מודלים עובדים טוב יותר עם חוקים פשוטים ולא עם איומים.
* רצפת ההזיות (The Hallucination Floor): בשום שלב סכנת ההזיות לא ירדה מ-11%. הסיבה לכך אינה קשורה לפרומפט, אלא לכשל בשליפה (Retrieval). מכיוון שב-12.7% מהשאלות המודל פשוט לא קיבל את מקטע הטקסט הנכון, הפרומפט לא יכול למנוע ממנו לנסות "להסתדר" עם מה שיש.

5. הצעד הבא (Next Steps)
--------------------------------------------------
כדי לפרוץ את תקרת ה-62% בהתאמת התשובות ולהוריד את ההזיות לאפס, הנדסת פרומפטים מיצתה את עצמה. הצעד ההכרחי כעת הוא שדרוג מנגנון השליפה על ידי הוספת שלב ה-Reranker (לדוגמה: שליפת 20 תוצאות ודירוגן מחדש כדי להבטיח שהתוצאה המדויקת תוגש ל-LLM תמיד בחמישייה הראשונה).
